# Bert Score

In [6]:
import bert_score

# Define reference and candidate texts
references = [
    "The cat sat on the mat.",
    "The feline rested on the floor covering."
]

candidates = [
    "A cat was sitting on a mat.",
    "The cat was on the mat."
]

# Calculate BERTScore
P, R, F1 = bert_score.score(
    candidates,
    references,
    lang="en",
    model_type="roberta-large",
    num_layers=17,
    verbose=True
)

# Print results
for i, (p, r, f) in enumerate(zip(P, R, F1)):
    print(f"Example {i+1}:")
    print(f"  Precision: {p.item():.4f}")
    print(f"  Recall: {r.item():.4f}")
    print(f"  F1: {f.item():.4f}")
    print()

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

D:\Projects\env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shivam\.cache\huggingface\hub\models--roberta-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

: 

In [ ]:
import torch

from transformers import AutoTokenizer, AutoModel

def get_bert_embeddings(texts, model_name="bert-base-uncased"):

    # Load tokenizer and model

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModel.from_pretrained(model_name)

    # Move model to GPU if available

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model.to(device)

    # Process texts in batch

    encoded_input = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}

    # Get model output

    with torch.no_grad():

        outputs = model(**encoded_input)

    # Use embeddings from the last layer

    embeddings = outputs.last_hidden_state

    # Remove padding tokens

    attention_mask = encoded_input['attention_mask']

    embeddings = [emb[mask.bool()] for emb, mask in zip(embeddings, attention_mask)]

    return embeddings

# Example usage

texts = ["The cat sat on the mat.", "A cat was sitting on a mat."]

embeddings = get_bert_embeddings(texts)

print(f"Number of texts: {len(embeddings)}")

print(f"Shape of first text embeddings: {embeddings[0].shape}")

In [1]:
from safetensors.torch import load_file
import json

# Load the weights
weights = load_file(r"D:\Projects\IMPOLS\pretrained_models\gpt2\model.safetensors")
#load config
with open(r"D:\Projects\IMPOLS\pretrained_models\gpt2\config.json") as f:
    hf_config = json.load(f)

# print(hf_config)

In [ ]:
import torch

# Assumes:
# model = your custom GPT model (already loaded with weights if needed)
# tokenizer = tiktoken.get_encoding("gpt2")

def get_gpt_embeddings(texts, model, tokenizer):
    # Move model to GPU if available
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    # Encode texts
    encoded = [tokenizer.encode(text) for text in texts]

    # Pad batch
    max_len = max(len(seq) for seq in encoded)
    padded = []
    attention_mask = []

    for seq in encoded:
        pad_len = max_len - len(seq)
        padded.append(seq + [0] * pad_len)
        attention_mask.append([1] * len(seq) + [0] * pad_len)

    input_ids = torch.tensor(padded).to(device)
    attention_mask = torch.tensor(attention_mask).to(device)

    # Get embeddings
    with torch.no_grad():
        outputs = model(input_ids, return_embeddings=True)
        # shape = (batch, seq_len, emb_dim)

    # Remove padding tokens
    embeddings = [
        emb[mask.bool()]
        for emb, mask in zip(outputs, attention_mask)
    ]

    return embeddings


# Example usage
texts = [
    "The cat sat on the mat.",
    "A cat was sitting on a mat."
]

embeddings = get_gpt_embeddings(texts, model, tokenizer)

print(f"Number of texts: {len(embeddings)}")
print(f"Shape of first text embeddings: {embeddings[0].shape}")

In [2]:
from nltk.translate.bleu_score import sentence_bleu

# Define reference and candidate sentences
reference = ['The cat is on the mat.']
candidate = 'The cat is sitting on the mat.'

# Tokenize the sentences
reference_tokenized = [sentence.split() for sentence in reference]
candidate_tokenized = candidate.split()

# Calculate BLEU score
bleu_score = sentence_bleu(reference_tokenized, candidate_tokenized)

print(f"BLEU Score: {bleu_score:.4f}")

ModuleNotFoundError: No module named 'nltk'